# Vasopressors Uncertainty Quantification

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")

# ----------------------------------------
# 1. Load dataset
# ----------------------------------------
print("Loading data...")
df = pd.read_csv("/content/mds_ed.csv")
print("Shape:", df.shape)




# vasopressors(agent that raises blood pressure by constricting blood vessels)
vasopressors_columns = [i for i in df.columns if 'vasopressors' in i] # it is a binary class whether the patient has vasopressors or no
Heartrate_columns = [i for i in df.columns if 'heart' in i] # checking if missing any other features (included in vitals)
Bloodpressure_cols= [i for i in df.columns if 'blood' in i] # included in lab_values

ECG_cols = [i for i in df.columns if 'ecg_' in i]
ECG_cols # for ECG data there is only when an ECG was taken and which ECG it is within a hospital stay

df['deterioration_vasopressors'].unique()

# ----------------------------------------
# 2. Feature groups
# ----------------------------------------
demographics_columns = [c for c in df.columns if "demographics_" in c]
biometrics_columns   = [c for c in df.columns if "biometrics_" in c]
vitals_columns       = [c for c in df.columns if "vitals_" in c]
labvalues_columns    = [c for c in df.columns if "labvalues_" in c]

features = (
    demographics_columns +
    biometrics_columns +
    vitals_columns +
    labvalues_columns
)

print("Number of features:", len(features))

# ----------------------------------------
# 3. Target: vasopressors
# ----------------------------------------
target_col = "deterioration_vasopressors"

if target_col not in df.columns:
    raise ValueError(f"{target_col} not found in dataset")

# Keep only rows where label exists
df = df.dropna(subset=[target_col])

# Convert to integer (important for XGBoost)
df[target_col] = df[target_col].astype(int)

# ----------------------------------------
# 4. Train-set median imputation (replacing missing vls with the mean)
# ----------------------------------------
train_mask = df["general_strat_fold"].isin(range(0, 18))
medians = df.loc[train_mask, features].median()
df[features] = df[features].fillna(medians)

# ----------------------------------------
# 5. Train / Val / Test split (data split into 20 folds one for testing, one for val, rest for training )
# ----------------------------------------
train_df = df[df["general_strat_fold"].isin(range(0, 18))].reset_index(drop=True)
val_df   = df[df["general_strat_fold"] == 18].reset_index(drop=True)
test_df  = df[df["general_strat_fold"] == 19].reset_index(drop=True)

# Use first ECG only (same as your setup)
val_df  = val_df[val_df["general_ecg_no_within_stay"] == 0]
test_df = test_df[test_df["general_ecg_no_within_stay"] == 0]

x_train = train_df[features].values
x_val   = val_df[features].values
x_test  = test_df[features].values

y_train = train_df[target_col].values
y_val   = val_df[target_col].values
y_test  = test_df[target_col].values

print("\nShapes:")
print("Train:", x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)

# ----------------------------------------
# 6. Train XGBoost model
# ----------------------------------------
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=4
)

model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    verbose=False
)

# ----------------------------------------
# 7. Evaluate
# ----------------------------------------
y_pred = model.predict_proba(x_test)[:, 1]
auroc = roc_auc_score(y_test, y_pred)

print("\n===== RESULT =====")
print("vasopressors AUROC:", round(auroc, 4))
print("Positive rate:", y_test.mean())

Loading data...
Shape: (4123, 1936)
Number of features: 470

Shapes:
Train: (3719, 470) Val: (193, 470) Test: (189, 470)

===== RESULT =====
vasopressors AUROC: 0.9681
Positive rate: 0.005291005291005291


array([0, 1])

In [ ]:
#Measures how close probabilities are to reality. Lower = better
from sklearn.metrics import brier_score_loss
brier = brier_score_loss(y_test, y_pred)
print("Brier score:", brier)

Brier score: 0.00544454404800475


**ICU 24h PREDICTION MODEL**

In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
import gc
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------
# 1. Load data
# ----------------------------------------
df = pd.read_csv('/content/mds_ed.csv')

# ----------------------------------------
# 2. Feature selection (REDUCED for memory safety)
# ----------------------------------------
features = (
    [c for c in df.columns if "vitals_" in c] +
    [c for c in df.columns if "labvalues_" in c] +
    [c for c in df.columns if "demographics_" in c] +
    [c for c in df.columns if "biometrics_" in c]
)

target_col = "deterioration_icu_24h"

# ----------------------------------------
# 3. Memory optimization (IMPORTANT)
# ----------------------------------------
df[features] = df[features].astype(np.float32)

# ----------------------------------------
# 4. Median imputation (train only)
# ----------------------------------------
train_mask = df["general_strat_fold"].isin(range(0, 18))
medians = df.loc[train_mask, features].median()
df[features] = df[features].fillna(medians)

# ----------------------------------------
# 5. Split dataset
# ----------------------------------------
train_df = df[df["general_strat_fold"].isin(range(0, 18))]
val_df   = df[df["general_strat_fold"] == 18]
test_df  = df[df["general_strat_fold"] == 19]

# Use only first ECG (early prediction setup)
train_df = train_df[train_df["general_ecg_no_within_stay"] == 0]
val_df   = val_df[val_df["general_ecg_no_within_stay"] == 0]
test_df  = test_df[test_df["general_ecg_no_within_stay"] == 0]

# ----------------------------------------
# 6. Convert to numpy WITHOUT extra copies
# ----------------------------------------
x_train = train_df[features].to_numpy(dtype=np.float32)
x_val   = val_df[features].to_numpy(dtype=np.float32)
x_test  = test_df[features].to_numpy(dtype=np.float32)

y_train = train_df[target_col].to_numpy()
y_val   = val_df[target_col].to_numpy()
y_test  = test_df[target_col].to_numpy()

# ----------------------------------------
# 7. Clean labels (remove NaN safely)
# ----------------------------------------
def clean(x, y):
    mask = (y != -999) & (~np.isnan(y))
    return x[mask], y[mask].astype(int)

x_train, y_train = clean(x_train, y_train)
x_val, y_val     = clean(x_val, y_val)
x_test, y_test   = clean(x_test, y_test)

# free memory
del df, train_df, val_df, test_df
gc.collect()

# ----------------------------------------
# 8. Train model (memory-optimized XGBoost)
# ----------------------------------------
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.6,
    tree_method="hist",   # IMPORTANT: reduces RAM
    max_bin=256,
    eval_metric="logloss",
    random_state=42,
    n_jobs=2
)

model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)

# ----------------------------------------
# 9. Predictions
# ----------------------------------------
y_pred = model.predict_proba(x_test)[:, 1]

# ----------------------------------------
# 10. Performance metrics
# ----------------------------------------
auroc = roc_auc_score(y_test, y_pred)
auprc = average_precision_score(y_test, y_pred)
brier = brier_score_loss(y_test, y_pred)

# ----------------------------------------
# 11. Uncertainty metrics
# ----------------------------------------
eps = 1e-8

entropy = -(y_pred * np.log(y_pred + eps) +
            (1 - y_pred) * np.log(1 - y_pred + eps))

confidence = np.abs(y_pred - 0.5) * 2

prob_true, prob_pred = calibration_curve(y_test, y_pred, n_bins=10)

# ----------------------------------------
# 12. Final results
# ----------------------------------------
print("\n===== ICU 24h PREDICTION MODEL =====")
print("AUROC:", round(auroc, 4))
print("AUPRC:", round(auprc, 4))
print("Brier score:", round(brier, 4)) # measures how good your model’s predicted probabilities are compared to the actual outcomes.
# Brier Score=N1​i=1/N∑​(pi​−yi​)^2

print("\n===== UNCERTAINTY =====")
print("Mean entropy:", entropy.mean())
print("Mean confidence:", confidence.mean())

print("\nPositive rate:", y_test.mean())


===== ICU 24h PREDICTION MODEL =====
AUROC: 0.9065
AUPRC: 0.6759
Brier score: 0.0624

===== UNCERTAINTY =====
Mean entropy: 0.23125927
Mean confidence: 0.831013

Positive rate: 0.12064121632787969
